# ICMI Corpus — Quantitative Analysis of GAT-2 Annotations
### Notebook 3 of 3: Figures and Statistics

**Prerequisite:** Run `ICMI_NB2_Corpus.ipynb` first to produce
`csv_export/korpus_gesamt.csv`. This notebook reads exclusively from that file.

**Three analytical parts:**
- **Part 1** — Corpus-wide overview: pause distribution, verbal/non-verbal tokens, turn-latching
- **Part 2** — Longitudinal comparison: Münster German group (2013 vs. 2014)
- **Part 3** — Co-occurrence of GAT-2 features by L1 (ger vs. por)

All figures saved as PNG to `diagramme/`. All rates normalised per minute.

---


## Setup


In [ ]:
import os, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

os.makedirs("diagramme", exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 150, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": "--",
    "font.family": "sans-serif", "font.size": 11,
    "axes.titlesize": 13, "axes.labelsize": 11, "legend.fontsize": 10,
})

C = {
    "ger": "#4C72B0", "por": "#DD8452", "swe": "#55A868",
    "2013": "#4C72B0", "2014": "#DD8452",
    "verbal": "#4C72B0", "nonverbal": "#DD8452",
    "micro": "#4C72B0", "short": "#DD8452", "long": "#55A868", "nv_pause": "#C44E52",
    "D1w": "#4C72B0", "D2m": "#DD8452", "D3w": "#55A868",
    "latch": "#8172B3",
}

GAT2_FEATURES = {
    "stress (CAPS)":     r"[A-ZÄÖÜ]{2,}",
    "lengthening (::)": r":{2,}",
    "pitch rise (↑)":   r"↑",
    "pitch fall (↓)":   r"↓",
    "micropause (.)": r"\(\.\)",
    "short pause (--)":  r"\(-{1,2}\)",
    "accent (´)":        r"[´`ˆˊˋ]",
    "boundary (,/;/.)": r"[,;]\s*$|(?<![A-Z])\.\s*$",
}

print("Setup complete.")


### Load corpus and derive recording durations

Post-processing applied here:
1. **L1 = ger** assigned to Münster German files, identified by speaker-ID pattern (`D\d[wmu]`) — works regardless of how the filenames are encoded locally.
2. **Recording duration** derived from the maximum `segment_ende` timestamp per file.


In [ ]:
corpus = pd.read_csv("csv_export/korpus_gesamt.csv", encoding="utf-8-sig")
print(f"Loaded: {len(corpus):,} rows · {corpus['datei'].nunique()} files")
print(f"Columns: {list(corpus.columns)}")

# Normalise sprecher: strip appended city/location names stored by NB2
# e.g. "D2m Warschau" → "D2m", "MGP" → "MGP", "B1" → "B1"
corpus["sprecher"] = corpus["sprecher"].apply(
    lambda x: str(x).strip().split()[0] if pd.notna(x) else x
)

# ── Identify Münster German files by speaker-ID pattern ───────────────────
# Speaker IDs like D1w, D2m, D3w indicate L1 German speakers.
# This avoids relying on filename encoding (Muenster / Mu_nster / Münster).
DE_SPK = r"^D\d[wmu]$"
munster_ger_files = set(
    corpus[
        corpus["sprecher"].str.match(DE_SPK, na=False) &
        corpus["datei"].str.match(r"^\d{4}", na=False)
    ]["datei"].unique()
)
print(f"\nMünster German files detected ({len(munster_ger_files)}):")
for f in sorted(munster_ger_files): print(f"  {f}")

corpus.loc[corpus["datei"].isin(munster_ger_files), "l1"] = "ger"

# Normalise L1 codes
L1_MAP = {"por":"por","pt":"por","deu":"ger","de":"ger","ale":"ger","swe":"swe","sv":"swe"}
corpus["l1"] = corpus["l1"].apply(
    lambda x: L1_MAP.get(str(x).lower().strip(), str(x).lower().strip())
)

# Duration from segment_ende (format MM:SS.s)
def ts_to_sek(ts):
    if pd.isna(ts): return np.nan
    m = re.match(r"(\d+):(\d+\.\d+)", str(ts))
    return int(m.group(1))*60 + float(m.group(2)) if m else np.nan

corpus["ende_sek"] = corpus["segment_ende"].apply(ts_to_sek)
dur_map = corpus.groupby("datei")["ende_sek"].max()
corpus["dur_min"] = corpus["datei"].map(dur_map) / 60

print(f"\nL1 distribution (verbal utterances):")
print(corpus[corpus["kanal"]=="v"].groupby("l1").size().sort_values(ascending=False))
print(f"\nRecording durations:")
for d, dur in dur_map.sort_index().items():
    print(f"  {d:<65} {dur/60:5.1f} min")


---
## Part 1 — Corpus-Wide Quantitative Overview


### 1.1 Pause Distribution

GAT-2 encodes pauses at three levels within verbal turns, plus a non-verbal channel:

| Symbol | Type | Description |
|---|---|---|
| `(.)` | Micropause | Very brief, within an intonation phrase |
| `(--)` | Short pause | One or two dashes |
| `(---)` | Long pause | Three or more dashes |
| `[nv]` | Non-verbal | Dedicated channel, no speaker attribution |


In [ ]:
verbal = corpus[corpus["kanal"]=="v"]

pause_counts = {
    "Micropause (.)": int(verbal["text"].str.count(r"\(\.\)").sum()),
    "Short pause (--)": int(verbal["text"].str.count(r"\(-{1,2}\)").sum()),
    "Long pause (---)": int(verbal["text"].str.count(r"\(-{3,}\)").sum()),
    "Non-verbal [nv]": int((corpus["kanal"]=="nv").sum()),
}

labels = list(pause_counts.keys())
values = list(pause_counts.values())
colors = [C["micro"], C["short"], C["long"], C["nv_pause"]]

fig, ax = plt.subplots(figsize=(7,7))
wedges, _, autotexts = ax.pie(
    values, colors=colors, explode=[0.03]*4,
    autopct=lambda p: f"{p:.1f}%" if p > 3 else "",
    startangle=140, pctdistance=0.75,
    wedgeprops={"linewidth": 1.5, "edgecolor": "white"},
)
for at in autotexts:
    at.set_fontsize(10); at.set_fontweight("bold")
ax.legend(wedges, [f"{l}  (n\u202f=\u202f{v:,})" for l,v in zip(labels,values)],
          loc="lower center", bbox_to_anchor=(0.5,-0.12), ncol=2, frameon=False)
ax.set_title("Pause Distribution across the ICMI Corpus", pad=18, fontweight="bold")
ax.text(0, 0, f"n = {sum(values):,}", ha="center", va="center", fontsize=12, color="#444")
plt.tight_layout()
plt.savefig("diagramme/1_1_pause_distribution.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/1_1_pause_distribution.png")


### 1.2 Verbal Tokens vs. Non-Verbal Actions per Speaker

- **Verbal tokens**: word count of transcribed speech (`[v]` channel)
- **Non-verbal actions**: `((...))` annotations within verbal turns plus `[nv]` lines

Only speakers with ≥ 50 utterances are shown. Values normalised per minute.


In [ ]:
spk_rows = []
for (spk, datei), grp in corpus.groupby(["sprecher","datei"]):
    dur = grp["dur_min"].iloc[0]
    if not dur or dur == 0: continue
    v = grp[grp["kanal"]=="v"]
    if len(v) < 50: continue
    spk_rows.append({
        "label": f"{spk}\n({grp['l1'].iloc[0]})",
        "tokens/min": v["text"].apply(lambda t: len(str(t).split())).sum() / dur,
        "nv/min": (v["text"].str.count(r"\(\(.+?\)\)").sum() +
                   len(grp[grp["kanal"]=="nv"])) / dur,
    })
spk_df = pd.DataFrame(spk_rows).sort_values("tokens/min", ascending=False)

fig, ax = plt.subplots(figsize=(14,6))
x = np.arange(len(spk_df))
ax.bar(x, spk_df["tokens/min"], color=C["verbal"],    alpha=0.85, label="Verbal tokens / min")
ax.bar(x, spk_df["nv/min"],     color=C["nonverbal"], alpha=0.85, label="Non-verbal actions / min",
       bottom=spk_df["tokens/min"])
ax.set_xticks(x)
ax.set_xticklabels(spk_df["label"], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Count per minute")
ax.set_title("Verbal Tokens and Non-Verbal Actions per Speaker (per minute)", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("diagramme/1_2_verbal_nonverbal_per_speaker.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/1_2_verbal_nonverbal_per_speaker.png")


### 1.3 Turn-Latching per Interview

Turn-latching (`=`) marks the immediate continuation of one speaker's turn
by another without any perceptible pause — a strong indicator of tight
sequential organisation.

The bar chart shows the latching rate (utterances containing `=` per minute)
for each interview file.

> **Note:** GAT-2 overlap markers (`[`) could not be reliably recovered from
> the PDF transcripts due to text-extraction artefacts. Latching serves as a
> related measure of interactional density.


In [ ]:
def short_label(s):
    s = re.sub(r"^(\d{4})_", r"\1 ", s)
    s = re.sub(r"Mu.?nster",     "Münster", s)
    s = re.sub(r"Muenster",      "Münster", s)
    s = re.sub(r"BeloHorizonte", "BH",       s)
    s = re.sub(r"Alema.?[eo]s?","DE",        s)
    s = re.sub(r"Brasileiros?",  "BR",        s)
    return re.sub(r"_", " ", s)[:32]

latch_rows = []
for datei, grp in corpus[corpus["kanal"]=="v"].groupby("datei"):
    dur = grp["dur_min"].iloc[0]
    if not dur or dur == 0: continue
    total  = len(grp)
    # Latching: = at end of utterance OR = at start (incoming latch)
    latch  = grp["text"].str.contains(r"=\s*$|^\s*=", na=False).sum()
    latch_rows.append({
        "label":      short_label(datei),
        "latch/min":  latch / dur,
        "latch_%":    latch / total * 100,
    })

latch_df = pd.DataFrame(latch_rows).sort_values("latch/min", ascending=False)

fig, ax = plt.subplots(figsize=(13,5))
x = np.arange(len(latch_df))
bars = ax.bar(x, latch_df["latch/min"], color=C["latch"], alpha=0.85)

for bar, (_, row) in zip(bars, latch_df.iterrows()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
            f"{row['latch_%']:.1f}%", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(latch_df["label"], rotation=40, ha="right", fontsize=8)
ax.set_ylabel("Latched utterances per minute")
ax.set_title("Turn-Latching (=) per Interview (per minute)\n"
             "Percentage labels show share of all utterances", fontweight="bold")
plt.tight_layout()
plt.savefig("diagramme/1_3_latching_per_interview.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/1_3_latching_per_interview.png")


---
## Part 2 — Longitudinal Comparison: Münster German Group (2013 vs. 2014)

D1w, D2m, and D3w participated in two sessions one year apart.
Both years consist of two files (Parte1 + Parte2), merged here.

> **Note on D3w:** City suffix changes between years (Tours 2013 → Stockholm 2014).
> Whether this is the same individual requires verification in the corpus documentation.


In [ ]:
CORE_SPEAKERS = {"D1w", "D2m", "D3w"}

# Use the set identified in the loading cell
munster = corpus[corpus["datei"].isin(munster_ger_files)].copy()
munster["year"] = munster["datei"].apply(
    lambda d: "2013" if d.startswith("2013") else "2014"
)

dur_per_year = (
    munster.drop_duplicates("datei")
    .groupby("year")["dur_min"].sum()
    .to_dict()
)

munster_kern = munster[munster["sprecher"].isin(CORE_SPEAKERS)]

print("Recording durations per year:")
for y, d in sorted(dur_per_year.items()): print(f"  {y}: {d:.1f} min")
print("\nCore-speaker verbal utterances:")
for year in ["2013","2014"]:
    v = munster_kern[(munster_kern["year"]==year)&(munster_kern["kanal"]=="v")]
    print(f"  {year}: {len(v)} total — " +
          ", ".join(f"{spk}={len(v[v['sprecher']==spk])}" for spk in sorted(CORE_SPEAKERS)))

# Safety check
for year in ["2013","2014"]:
    if dur_per_year.get(year, 0) == 0:
        print(f"  WARNING: no duration found for {year} — "
              f"verify that the correct files are in the corpus.")


### 2.1 Speaker Participation: Share and Utterance Rate


In [ ]:
spk_list = sorted(CORE_SPEAKERS)
x, width = np.arange(len(spk_list)), 0.35
fig, axes = plt.subplots(1, 2, figsize=(12,5))

for ax, (metric, ylabel, title) in zip(axes, [
    ("share", "Share of core-speaker utterances (%)", "Speaker Share (%)"),
    ("rate",  "Utterances per minute",                 "Utterance Rate (per min)"),
]):
    for i, year in enumerate(["2013","2014"]):
        grp_v = munster_kern[(munster_kern["year"]==year)&(munster_kern["kanal"]=="v")]
        total = len(grp_v)
        dur   = dur_per_year.get(year, 1)
        if total == 0 or dur == 0:
            print(f"WARNING: no data for {year} — skipping.")
            continue
        vals = []
        for spk in spk_list:
            n = len(grp_v[grp_v["sprecher"]==spk])
            vals.append(n/total*100 if metric=="share" else n/dur)
        bars = ax.bar(x+(i-0.5)*width, vals, width, label=year, color=C[year], alpha=0.85)
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                    f"{val:.1f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(spk_list)
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight="bold")
    ax.legend(frameon=False)

fig.suptitle("Speaker Participation — Münster German Group (2013 vs. 2014)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("diagramme/2_1_speaker_participation.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/2_1_speaker_participation.png")


### 2.2 GAT-2 Feature Profile — Heatmap

Each cell: % of utterances by that speaker/year containing the feature at least once.


In [ ]:
feat_labels = list(GAT2_FEATURES.keys())
spk_list    = sorted(CORE_SPEAKERS)
years       = ["2013","2014"]
col_labels  = [f"{s}\n{y}" for y in years for s in spk_list]

matrix = np.zeros((len(feat_labels), len(col_labels)))
for ci, (year, spk) in enumerate([(y,s) for y in years for s in spk_list]):
    grp = munster_kern[
        (munster_kern["year"]==year) &
        (munster_kern["sprecher"]==spk) &
        (munster_kern["kanal"]=="v")
    ]
    n = len(grp)
    for fi, (feat, regex) in enumerate(GAT2_FEATURES.items()):
        if n > 0:
            matrix[fi,ci] = grp["text"].apply(lambda t: bool(re.search(regex,str(t)))).sum()/n*100

fig, ax = plt.subplots(figsize=(11,6))
im = ax.imshow(matrix, aspect="auto", cmap="YlOrRd", vmin=0)
ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=9)
ax.set_yticks(range(len(feat_labels)))
ax.set_yticklabels(feat_labels, fontsize=10)
ax.axvline(len(spk_list)-0.5, color="white", linewidth=3)
ax.text(len(spk_list)/2-0.5,  -1.2, "2013", ha="center", fontsize=11, fontweight="bold")
ax.text(len(spk_list)*1.5-0.5,-1.2, "2014", ha="center", fontsize=11, fontweight="bold")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(j, i, f"{matrix[i,j]:.0f}%", ha="center", va="center", fontsize=8,
                color="white" if matrix[i,j]>40 else "#333")
plt.colorbar(im, ax=ax, label="% utterances containing feature", shrink=0.8)
ax.set_title("GAT-2 Feature Profile per Speaker and Year — Münster German Group",
             fontweight="bold", pad=22)
plt.tight_layout()
plt.savefig("diagramme/2_2_feature_heatmap.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/2_2_feature_heatmap.png")


### 2.3 Lengthening as a Planning Marker

A decline in lengthening (`::`) alongside longer utterances may indicate
increased automaticity in L2 production.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
for ax, (use_regex, ylabel, title) in zip(axes, [
    (True,  "% utterances with lengthening (::)", "Lengthening Rate"),
    (False, "Mean utterance length (tokens)",      "Mean Utterance Length"),
]):
    for spk in sorted(CORE_SPEAKERS):
        vals = []
        for year in ["2013","2014"]:
            grp = munster_kern[
                (munster_kern["year"]==year) &
                (munster_kern["sprecher"]==spk) &
                (munster_kern["kanal"]=="v")
            ]
            n = len(grp)
            if use_regex:
                v = grp["text"].apply(lambda t: bool(re.search(r":{2,}",str(t)))).sum()/n*100 if n else 0
            else:
                v = grp["text"].apply(lambda t: len(str(t).split())).mean() if n else 0
            vals.append(v)
        ax.plot(["2013","2014"], vals, marker="o", linewidth=2, color=C[spk], label=spk)
        ax.text("2014", vals[1]+0.1, f"{vals[1]:.1f}", fontsize=9, color=C[spk])
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight="bold")
    ax.legend(frameon=False)
    ax.set_ylim(bottom=0)

fig.suptitle("Lengthening and Utterance Length — Münster German Group",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("diagramme/2_3_lengthening_utterance_length.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/2_3_lengthening_utterance_length.png")


### 2.4 Pause Profile per Speaker and Year


In [ ]:
PAUSE_TYPES = {
    "Micropause (.)": (r"\(\.\)",    C["micro"]),
    "Short pause (--)": (r"\(-{1,2}\)",C["short"]),
    "Long pause (---)": (r"\(-{3,}\)", C["long"]),
}
spk_list = sorted(CORE_SPEAKERS)
x = np.arange(len(spk_list))
fig, axes = plt.subplots(1, 2, figsize=(13,5), sharey=True)

for ax, year in zip(axes, ["2013","2014"]):
    dur    = dur_per_year.get(year, 1)
    bottom = np.zeros(len(spk_list))
    for ptype, (regex, color) in PAUSE_TYPES.items():
        vals = []
        for spk in spk_list:
            grp = munster_kern[
                (munster_kern["year"]==year) &
                (munster_kern["sprecher"]==spk) &
                (munster_kern["kanal"]=="v")
            ]
            vals.append(grp["text"].str.count(regex).sum() / dur)
        ax.bar(x, vals, label=ptype, color=color, alpha=0.85, bottom=bottom)
        bottom += np.array(vals)
    ax.set_xticks(x); ax.set_xticklabels(spk_list)
    ax.set_title(year, fontweight="bold"); ax.set_xlabel("Speaker")

axes[0].set_ylabel("Pauses per minute")
axes[1].legend(frameon=False)
fig.suptitle("Pause Profile per Speaker — Münster German Group (2013 vs. 2014)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("diagramme/2_4_pause_profile.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/2_4_pause_profile.png")


---
## Part 3 — Co-Occurrence Analysis by L1

Analysis focuses on **ger** vs. **por** — the two groups with sufficient data.
All utterances with unknown L1 are excluded.


### 3.1 Tokens, Non-Verbal Actions and Pauses by L1 (per minute)


In [ ]:
L1_GROUPS = ["ger","por"]
l1_stats  = {}
for l1 in L1_GROUPS:
    grp_v  = corpus[(corpus["l1"]==l1)&(corpus["kanal"]=="v")]
    grp_nv = corpus[(corpus["l1"]==l1)&(corpus["kanal"]=="nv")]
    dur    = corpus[corpus["l1"]==l1].drop_duplicates("datei")["dur_min"].sum()
    l1_stats[l1] = {
        "Tokens / min":              grp_v["text"].apply(lambda t: len(str(t).split())).sum()/dur,
        "Non-verbal\nactions / min": (grp_v["text"].str.count(r"\(\(.+?\)\)").sum()+len(grp_nv))/dur,
        "Pauses / min":              grp_v["text"].str.count(r"\(\.?-*\)").sum()/dur,
    }

metrics = list(next(iter(l1_stats.values())).keys())
x, width = np.arange(len(metrics)), 0.3
fig, ax = plt.subplots(figsize=(9,5))
for i, l1 in enumerate(L1_GROUPS):
    vals = [l1_stats[l1][m] for m in metrics]
    bars = ax.bar(x+(i-0.5)*width, vals, width, label=f"L1 {l1}", color=C[l1], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                f"{val:.1f}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel("Rate per minute")
ax.set_title("Tokens, Non-Verbal Actions and Pauses by L1 (per minute)", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("diagramme/3_1_l1_rates.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/3_1_l1_rates.png")


### 3.2 Feature Co-Occurrence Matrix by L1

Each cell: % of utterances in which both features co-occur.
L1 ger and L1 por shown side by side.


In [ ]:
feat_labels = list(GAT2_FEATURES.keys())
n_feats     = len(feat_labels)
fig, axes   = plt.subplots(1, 2, figsize=(16,6))

for ax, l1 in zip(axes, L1_GROUPS):
    grp = corpus[(corpus["l1"]==l1)&(corpus["kanal"]=="v")].copy()
    for feat, regex in GAT2_FEATURES.items():
        grp[feat] = grp["text"].apply(lambda t: int(bool(re.search(regex,str(t)))))
    n = len(grp)
    matrix = np.zeros((n_feats, n_feats))
    for i, f1 in enumerate(feat_labels):
        for j, f2 in enumerate(feat_labels):
            matrix[i,j] = ((grp[f1]==1)&(grp[f2]==1)).sum()/n*100 if n else 0
    im = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=matrix.max())
    ax.set_xticks(range(n_feats)); ax.set_yticks(range(n_feats))
    ax.set_xticklabels(feat_labels, rotation=40, ha="right", fontsize=9)
    ax.set_yticklabels(feat_labels, fontsize=9)
    for i in range(n_feats):
        for j in range(n_feats):
            ax.text(j, i, f"{matrix[i,j]:.0f}%", ha="center", va="center",
                    fontsize=7, color="white" if matrix[i,j]>matrix.max()*0.6 else "#333")
    plt.colorbar(im, ax=ax, label="% of utterances", shrink=0.8)
    ax.set_title(f"L1 = {l1}  (n = {n:,} utterances)", fontweight="bold", pad=12)

fig.suptitle("GAT-2 Feature Co-Occurrence Matrix by L1 (ger vs. por)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("diagramme/3_2_cooccurrence_matrix.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/3_2_cooccurrence_matrix.png")


### 3.3 Top 5 Co-Occurring Feature Pairs by L1

The five strongest co-occurrence pairs (excluding self-pairs)
as a grouped bar chart for direct comparison.


In [ ]:
pair_rates = {l1:{} for l1 in L1_GROUPS}
for l1 in L1_GROUPS:
    grp = corpus[(corpus["l1"]==l1)&(corpus["kanal"]=="v")].copy()
    for feat, regex in GAT2_FEATURES.items():
        grp[feat] = grp["text"].apply(lambda t: int(bool(re.search(regex,str(t)))))
    n = len(grp)
    for i, f1 in enumerate(feat_labels):
        for j, f2 in enumerate(feat_labels):
            if j <= i: continue
            key = f"{f1}\n+ {f2}"
            pair_rates[l1][key] = ((grp[f1]==1)&(grp[f2]==1)).sum()/n*100 if n else 0

all_pairs = set(pair_rates["ger"])|set(pair_rates["por"])
avg  = {p: sum(pair_rates[l1].get(p,0) for l1 in L1_GROUPS)/2 for p in all_pairs}
top5 = sorted(avg, key=avg.get, reverse=True)[:5]

fig, ax = plt.subplots(figsize=(11,5))
x, width = np.arange(len(top5)), 0.35
for i, l1 in enumerate(L1_GROUPS):
    vals = [pair_rates[l1].get(p,0) for p in top5]
    bars = ax.bar(x+(i-0.5)*width, vals, width, label=f"L1 {l1}", color=C[l1], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(top5, fontsize=9)
ax.set_ylabel("% of utterances")
ax.set_title("Top 5 Co-Occurring GAT-2 Feature Pairs by L1", fontweight="bold")
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig("diagramme/3_3_top_cooccurrence_pairs.png", bbox_inches="tight")
plt.show()
print("Saved: diagramme/3_3_top_cooccurrence_pairs.png")


---
## Summary of Outputs

| File | Content |
|---|---|
| `1_1_pause_distribution.png` | Pie: pause types, full corpus |
| `1_2_verbal_nonverbal_per_speaker.png` | Stacked bar: tokens vs. non-verbal actions / min |
| `1_3_latching_per_interview.png` | Bar: turn-latching rate per interview |
| `2_1_speaker_participation.png` | Grouped bar: speaker share and rate 2013 vs. 2014 |
| `2_2_feature_heatmap.png` | Heatmap: GAT-2 features per speaker and year |
| `2_3_lengthening_utterance_length.png` | Line: lengthening rate and utterance length |
| `2_4_pause_profile.png` | Stacked bar: pause types per speaker, both years |
| `3_1_l1_rates.png` | Grouped bar: tokens / non-verbals / pauses by L1 |
| `3_2_cooccurrence_matrix.png` | Heatmap: feature co-occurrence ger vs. por |
| `3_3_top_cooccurrence_pairs.png` | Grouped bar: top 5 feature pairs by L1 |
